<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [1]:
%load_ext autoreload
%autoreload 2

### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan perhitungan fitur density

In [2]:
# Memilih dataset dan parameter pemisahan
import pandas as pd
from IPython.display import display, Markdown
from pci_ml.data import list_datasets, load_data, clean_data, split_data
from pci_ml.memory import configure, release
from pci_ml.ui import build_panel

EXPORT_MODE = True   # True = print panel values instead of showing widgets
RELEASE_MEMORY = True

DEFAULTS = {
    "file": None,        # None = first dataset, or e.g. "data_2026.xlsx"
    "n_seed": 200,
    "test_size": 0.2,
    "share_tol": 0.01,
    "pci_min": 0,
}

configure(release=RELEASE_MEMORY)

datasets = list_datasets()
if DEFAULTS["file"] is not None and DEFAULTS["file"] not in datasets:
    raise ValueError(f"DEFAULTS['file'] '{DEFAULTS['file']}' not found in datasets")

# Helper function: compact preview of the first rows
def format_adaptive_df(df_input):
    if df_input is None:
        return ""
    df_head = df_input.head()
    left_count = 5
    right_count = 2
    if df_head.shape[1] > (left_count + right_count):
        left_cols = df_head.iloc[:, :left_count]
        right_cols = df_head.iloc[:, -right_count:]
        middle_col = pd.DataFrame({"…": ["…"] * len(df_head)}, index=df_head.index)
        df_display = pd.concat([left_cols, middle_col, right_cols], axis=1)
    else:
        df_display = df_head
    markdown_table = df_display.to_markdown(index=False)
    return f'<div style="font-size: 11px;">\n\n{markdown_table}\n\n</div>'

df = None
handle = None

def on_select(name):
    raw = None if name is None else load_data(name)
    handle.update(Markdown(format_adaptive_df(raw)))

panel = build_panel(datasets, DEFAULTS, on_file_change=on_select, export=EXPORT_MODE)

handle = display(Markdown(""), display_id=True)

on_select(panel.values()["file"])


File terpilih untuk analisis: pci-ml-26.xlsx
Parameter Pemisahan Data:
- Jumlah seed     : 200
- Test size       : 0.2
- Share tolerance : 0.01
- PCI minimum     : 0.0


<div style="font-size: 11px;">

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   SEGMENT_AREA |   GATOR_CRACK_L | …   |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|---------------:|----------------:|:----|--------------:|--------:|
| Guru Bangkol | R001      | S001         |            450 |            0    | …   |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |            450 |            0    | …   |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |            450 |            0    | …   |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |            450 |            0.64 | …   |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |            450 |            0    | …   |             0 | 96.1874 |

</div>

In [3]:
# Membersihkan data dari duplikat, pencilan dan data tidak valid, serta menghitung fitur density
cfg = panel.values()
filename = cfg["file"]
df, features = clean_data(load_data(filename))

if not EXPORT_MODE:
    print(f"Parameter: file {filename} | jumlah seed {cfg['n_seed']} | "
          f"test size {cfg['test_size']} | toleransi {cfg['share_tol']} | "
          f"PCI minimum {cfg['pci_min']}")

markdown_table = format_adaptive_df(df)
print("Data telah dibersihkan dan fitur densitas telah dihitung.")
display(Markdown(markdown_table))


Data telah dibersihkan dan fitur densitas telah dihitung.


<div style="font-size: 11px;">

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   GATOR_CRACK_L |   GATOR_CRACK_M | …   |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|----------------:|----------------:|:----|--------------:|--------:|
| Guru Bangkol | R001      | S001         |      0          |      0          | …   |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |      0          |      0.00391778 | …   |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |      0          |      0          | …   |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |      0.00142222 |      0.0231111  | …   |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |      0          |      0          | …   |             0 | 96.1874 |

</div>

#### 2. Pemisahan data latih dan data uji

In [4]:
# Memisah data sesuai kriteria
from IPython.display import clear_output

if globals().get("df") is None:
    df, features = clean_data(load_data(filename))

def show_progress(done, total, best_gap):
    if done % 100 != 0 and done != total:
        return
    gap = "-" if best_gap is None else f"{best_gap:.4f}"
    clear_output(wait=True)
    print(f"Mencari Seeds: {done}/{total} | Best Gap: {gap}", flush=True)

result = split_data(df, features, filename, {
    "n_seed": cfg["n_seed"],
    "test_size": cfg["test_size"],
    "share_tol": cfg["share_tol"],
    "pci_min": cfg["pci_min"],
}, progress=show_progress)
print()

split_summary = result["summary"]

# Print the summary outside the progress line so it always shows up
print(split_summary)

release("result", "df", "show_progress")


Mencari Seeds: 200/200 | Best Gap: 0.7556

Seed used: 179 (Gap: 0.7556)
Train: 933 rows (79.27%) | Test: 244 rows (20.73%)
Roads: 27 train | 7 test
Saved to: "C:\Users\gearb\Desktop\pci-ml-single\pci-ml\datasets\pci-ml-26\n200_t0.2_s0.01_p0.0"


#### 3. Pelatihan model *machine learning*

In [5]:
# Mempersiapkan input model
import numpy as np
from pci_ml.data import PCI, ROAD_ID, METADATA, latest_checkpoint, load_checkpoint

SEED = 42   # model seed (RF, XGBoost, Keras), separate from the split seed in meta["seed"]

# Memuat checkpoint terbaru untuk file yang dipilih
ckpt_tag = latest_checkpoint(filename)

X_train = X_test = y_train = y_test = groups_train = test_ids = meta = None

def load_latest_ckpt(tag):
    global X_train, X_test, y_train, y_test, groups_train, test_ids, features, meta
    # train_df / test_df are local: they are freed when this function returns
    train_df, test_df, features, meta = load_checkpoint(filename, tag)

    X_train = train_df[features].to_numpy(dtype="float32")
    X_test  = test_df[features].to_numpy(dtype="float32")
    y_train = train_df[PCI].to_numpy(dtype="float32")    # PCI scale 0-100; divide by 100 in the ANN cell
    y_test  = test_df[PCI].to_numpy(dtype="float32")

    groups_train = train_df[ROAD_ID].to_numpy()          # for GroupKFold when tuning
    test_ids = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)   # labels for result tables

    assert all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)), "NaN/inf found"

def split_info():
    n_tr, n_te = len(X_train), len(X_test)
    total = n_tr + n_te
    return (f"**Train:** {X_train.shape} ({n_tr / total:.2%}) | "
            f"**Test:** {X_test.shape} ({n_te / total:.2%})")

load_latest_ckpt(ckpt_tag)

# Staleness guard: the newest split may have been made with different parameters
current = {key: cfg[key] for key in ("n_seed", "test_size", "share_tol", "pci_min")}
if current != meta["params"]:
    print("PERINGATAN: split terbaru dibuat dengan parameter berbeda. "
          "Jalankan ulang Cell 7 untuk memisahkan data dengan parameter saat ini.")
    print(f"  Parameter saat ini : {current}")
    print(f"  Parameter split    : {meta['params']}")

display(Markdown(f"**Split terpilih:** `{ckpt_tag}`\n\n{split_info()}"))

release("df", "ckpt_tag", "load_latest_ckpt", "split_info", "current")


**Split terpilih:** `n200_t0.2_s0.01_p0.0`

**Train:** (933, 55) (79.27%) | **Test:** (244, 55) (20.73%)

In [12]:
# DINONAKTIFKAN SEMENTARA: verifikasi data input
# Verifikasi data input
import numpy as np
from pci_ml.data import ROAD_ID

# 1. Shapes and dtypes
print("Arrays:")
for name in ("X_train", "X_test", "y_train", "y_test", "groups_train"):
    a = globals()[name]
    print(f"  {name:<14}{str(a.shape):<12}{a.dtype}")
print(f"  {'test_ids':<14}{str(test_ids.shape):<12}columns: {list(test_ids.columns)}")
print(f"  {'features':<14}{len(features)} columns | first 3: {features[:3]}")

# 2. Consistency checks
checks = {
    "X columns == len(features)":        X_train.shape[1] == X_test.shape[1] == len(features),
    "X/y/groups rows match (train)":     len(X_train) == len(y_train) == len(groups_train),
    "X/y/test_ids rows match (test)":    len(X_test) == len(y_test) == len(test_ids),
    "row counts match meta":             (len(X_train), len(X_test)) == (meta["n_train"], meta["n_test"]),
    "no NaN/inf in arrays":              all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)),
    "X within 0-1":                      min(X_train.min(), X_test.min()) >= 0 and max(X_train.max(), X_test.max()) <= 1,
    "y within 0-100":                    min(y_train.min(), y_test.min()) >= 0 and max(y_train.max(), y_test.max()) <= 100,
    "no road shared by train and test":  set(groups_train).isdisjoint(test_ids[ROAD_ID]),
    "train_df/test_df released":         "train_df" not in globals() and "test_df" not in globals(),
}
print("\nChecks:")
for label, ok in checks.items():
    print(f"  [{'OK' if ok else 'FAIL'}] {label}")

# 3. Summary of the selected split
print(f"\nSeed split: {meta['seed']} | Gap: {meta['gap']:.4f} | Dibuat: {meta['created_at']}")
print(f"PCI train: mean {y_train.mean():.2f}, std {y_train.std(ddof=1):.2f} | "
      f"test: mean {y_test.mean():.2f}, std {y_test.std(ddof=1):.2f}")

Arrays:
  X_train       (933, 55)   float32
  X_test        (244, 55)   float32
  y_train       (933,)      float32
  y_test        (244,)      float32
  groups_train  (933,)      object
  test_ids      (244, 3)    columns: ['ROAD_ID', 'ROAD_NAME', 'SEGMENT_ID']
  features      55 columns | first 3: ['GATOR_CRACK_L', 'GATOR_CRACK_M', 'GATOR_CRACK_H']

Checks:
  [OK] X columns == len(features)
  [OK] X/y/groups rows match (train)
  [OK] X/y/test_ids rows match (test)
  [OK] row counts match meta
  [OK] no NaN/inf in arrays
  [OK] X within 0-1
  [OK] y within 0-100
  [OK] no road shared by train and test
  [OK] train_df/test_df released

Seed split: 179 | Gap: 0.7556 | Dibuat: 2026-11-08 | 00:39:41
PCI train: mean 68.37, std 22.92 | test: mean 68.32, std 22.20


In [7]:
# DINONAKTIFKAN SEMENTARA: baseline Random Forest.
# Cell ini tidak dipakai oleh tuning maupun model final (lihat cell tuning).
# Hapus tanda '#' di awal setiap baris untuk mengaktifkan kembali.

# # Melatih dan menguji model Random Forest
# from pci_ml.random_forest import train_random_forest, evaluate_random_forest
# from pci_ml.ui import build_param_panel
#
# RF_DEFAULTS = {
#     "n_estimators": 300,
#     "max_depth": 0,          # 0 = kedalaman tidak dibatasi
#     "min_samples_split": 2,
#     "min_samples_leaf": 1,
#     "max_features": 1.0,
#     "max_samples": 1.0,      # 1.0 = semua sampel
# }
#
# RF_SPECS = {
#     "n_estimators":      {"type": "int",   "description": "Jumlah pohon:", "label": "Jumlah pohon"},
#     "max_depth":         {"type": "int",   "description": "Kedalaman maks:", "label": "Kedalaman maks"},
#     "min_samples_split": {"type": "int",   "description": "Min sampel split:", "label": "Min sampel split"},
#     "min_samples_leaf":  {"type": "int",   "description": "Min sampel daun:", "label": "Min sampel daun"},
#     "max_features":      {"type": "float", "description": "Maks fitur:", "label": "Maks fitur"},
#     "max_samples":       {"type": "float", "description": "Maks sampel:", "label": "Maks sampel"},
# }
#
# rf_panel = build_param_panel(RF_DEFAULTS, RF_SPECS, "Parameter Random Forest:", export=EXPORT_MODE)
# rf_params = rf_panel.values()
#
# if rf_params["max_depth"] == 0:
#     print("Catatan: kedalaman 0 berarti tanpa batas.")
#
# model = train_random_forest(X_train, y_train, rf_params, random_state=SEED)
# train_metrics = evaluate_random_forest(model, X_train, y_train)
# test_metrics = evaluate_random_forest(model, X_test, y_test)
#
# print("\nHasil Random Forest:")
# print(f"- Data latih: MAE {train_metrics['mae']:.4f} | RMSE {train_metrics['rmse']:.4f} | R² {train_metrics['r2']:.4f}")
# print(f"- Data uji  : MAE {test_metrics['mae']:.4f} | RMSE {test_metrics['rmse']:.4f} | R² {test_metrics['r2']:.4f}")

In [13]:
# Tuning hyperparameter Random Forest dengan GroupKFold
from IPython.display import Markdown, display, HTML
from pci_ml.random_forest import tune_random_forest, evaluate_random_forest
from pci_ml.ui import build_param_panel

TUNING_DEFAULTS = {
    "n_iter": 100,
    "cv_splits": 5,
}

TUNING_SPECS = {
    "n_iter":    {"type": "int", "description": "Jumlah kandidat:", "label": "Jumlah kandidat"},
    "cv_splits": {"type": "int", "description": "Jumlah fold:", "label": "Jumlah fold"},
}

# RF_SEARCH_SPACE = {
#     "n_estimators": [300, 500],
#     "max_depth": [None, 40, 30, 20],
#     "min_samples_split": [2, 5, 10],
#     "min_samples_leaf": [1, 2, 3],
#     "max_features": [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0],
#     "max_samples": [None, 0.6, 0.7, 0.8, 0.9],
# }

RF_SEARCH_SPACE = {
    "n_estimators": [300],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2],
    "max_features": [0.1, 0.15, 0.2, 0.25, 0.3, 0.4],
    "max_samples": [0.5, 0.6, 0.7, 0.8, 0.9],
}

tuning_panel = build_param_panel(
    TUNING_DEFAULTS, TUNING_SPECS, "Parameter Tuning Random Forest:", export=EXPORT_MODE
)
tuning_params = tuning_panel.values()

display(Markdown(
    f"**Memulai tuning:** {tuning_params['n_iter']} kandidat | "
    f"{tuning_params['cv_splits']} fold CV"
))
rf_result = tune_random_forest(
    X_train, y_train, groups_train,
    RF_SEARCH_SPACE,
    n_iter=tuning_params["n_iter"],
    cv_splits=tuning_params["cv_splits"],
    random_state=SEED,
)

rf_best_params = rf_result["params"]
rf_cv = rf_result["cv_pooled"]
tuned_model = rf_result["model"]
tuned_test = evaluate_random_forest(tuned_model, X_test, y_test)

def format_value(value):
    return "tanpa batas" if value is None else value

param_explanations = {
    "n_estimators": "Jumlah pohon keputusan",
    "max_depth": "Kedalaman maksimum tiap pohon.",
    "min_samples_split": "Jumlah sampel minimum agar sebuah node boleh dipecah.",
    "min_samples_leaf": "Jumlah sampel minimum di tiap daun.",
    "max_features": "Jumlah fraksi fitur di tiap pohon.",
    "max_samples": "Jumlah data latih yang dipakai untuk tiap pohon.",
}

rows_html = ""
for key in RF_SEARCH_SPACE:
    val = format_value(rf_best_params[key])
    desc = param_explanations[key]
    rows_html += (
        "    <tr style='border-bottom: 1px solid #eee;'>\n"
        f"      <td style='padding: 6px 10px; font-family: monospace; white-space: nowrap;'>{key}</td>\n"
        f"      <td style='padding: 6px 10px; text-align: center;'>{val}</td>\n"
        f"      <td style='padding: 6px 10px; max-width: 400px; word-wrap: break-word;'>{desc}</td>\n"
        "    </tr>\n"
    )

param_table = (
    "<table style='font-size: 11px; border-collapse: collapse; width: 100%;'>\n"
    "  <thead>\n"
    "    <tr style='border-bottom: 2px solid #ddd;'>\n"
    "      <th style='padding: 6px 10px; text-align: left; white-space: nowrap;'>Hyperparameter</th>\n"
    "      <th style='padding: 6px 10px; text-align: center; white-space: nowrap;'>Nilai</th>\n"
    "      <th style='padding: 6px 10px; text-align: left;'>Keterangan</th>\n"
    "    </tr>\n"
    "  </thead>\n"
    "  <tbody>\n"
    + rows_html
    + "  </tbody>\n"
    "</table>"
)

metric_table = (
    "| Evaluasi | MAE | RMSE | R\u00b2 |\n"
    "|---|---:|---:|---:|\n"
    f"| CV pooled OOF | {rf_cv['mae']:.4f} | {rf_cv['rmse']:.4f} | {rf_cv['r2']:.4f} |\n"
    f"| Test | {tuned_test['mae']:.4f} | {tuned_test['rmse']:.4f} | {tuned_test['r2']:.4f} |\n"
)

n_test_roads = test_ids["ROAD_ID"].nunique()
explanation = f"""
**Keterangan**
- **MAE**: rata-rata selisih absolut antara prediksi dan PCI aktual, dalam satuan poin PCI. Semakin kecil semakin baik.
- **RMSE**: akar rata-rata kuadrat error, dalam satuan poin PCI. Lebih sensitif terhadap error besar; semakin kecil semakin baik.
- **R\u00b2**: proporsi variasi PCI yang dijelaskan model. Semakin mendekati 1 semakin baik.
- **CV pooled OOF**: semua prediksi out-of-fold dari {tuning_params['cv_splits']} fold digabung, lalu dihitung sekali. Ini estimasi generalisasi yang lebih stabil daripada rata-rata R\u00b2 per fold, tetapi sedikit optimistis karena hyperparameter dipilih dari CV yang sama.
- **Test**: evaluasi pada data uji yang tidak dipakai saat tuning. Ini estimasi akhir pada {n_test_roads} road holdout, tetapi hanya satu holdout sehingga variansnya lebih tinggi.
"""

display(Markdown("### Hyperparameter terbaik"))
display(HTML(param_table))
display(Markdown(f"### Hasil evaluasi\n\n{metric_table}\n\n{explanation}"))


Parameter Tuning Random Forest:
- Jumlah kandidat   : 100
- Jumlah fold       : 5


**Memulai tuning:** 100 kandidat | 5 fold CV

### Hyperparameter terbaik

Hyperparameter,Nilai,Keterangan
n_estimators,300,Jumlah pohon keputusan
max_depth,tanpa batas,Kedalaman maksimum tiap pohon.
min_samples_split,2,Jumlah sampel minimum agar sebuah node boleh dipecah.
min_samples_leaf,1,Jumlah sampel minimum di tiap daun.
max_features,0.2,Jumlah fraksi fitur di tiap pohon.
max_samples,0.9,Jumlah data latih yang dipakai untuk tiap pohon.


### Hasil evaluasi

| Evaluasi | MAE | RMSE | R² |
|---|---:|---:|---:|
| CV pooled OOF | 7.1919 | 11.3480 | 0.7545 |
| Test | 5.7527 | 8.1144 | 0.8659 |



**Keterangan**
- **MAE**: rata-rata selisih absolut antara prediksi dan PCI aktual, dalam satuan poin PCI. Semakin kecil semakin baik.
- **RMSE**: akar rata-rata kuadrat error, dalam satuan poin PCI. Lebih sensitif terhadap error besar; semakin kecil semakin baik.
- **R²**: proporsi variasi PCI yang dijelaskan model. Semakin mendekati 1 semakin baik.
- **CV pooled OOF**: semua prediksi out-of-fold dari 5 fold digabung, lalu dihitung sekali. Ini estimasi generalisasi yang lebih stabil daripada rata-rata R² per fold, tetapi sedikit optimistis karena hyperparameter dipilih dari CV yang sama.
- **Test**: evaluasi pada data uji yang tidak dipakai saat tuning. Ini estimasi akhir pada 7 road holdout, tetapi hanya satu holdout sehingga variansnya lebih tinggi.


#### 4. Perbaikan model Random Forest

Diagnostik menunjukkan model *Random Forest* (RF) menyusutkan prediksi ke arah rata-rata: jalan dengan PCI rendah diprediksi terlalu tinggi, sedangkan jalan dengan PCI tinggi sedikit diprediksi terlalu rendah. Bagian ini menguji empat perbaikan pada lipatan (*fold*) yang sama.

1. **Phase 0** - harness evaluasi dan audit data.
2. **Phase 1** - fitur agregat (total density dan *severity-weighted density*).
3. **Phase 2** - bobot sampel yang menekankan kondisi buruk.
4. **Phase 3** - kalibrasi prediksi *post-hoc*.
5. **Phase 4** - perbandingan, uji ketahanan, dan rekomendasi.

Pemilihan konfigurasi hanya memakai CV (*out-of-fold*), bukan data uji.

##### 4.1 Phase 0 - Harness evaluasi dan audit data

Semua eksperimen memakai `FOLDS` yang sama, dihitung sekali dari `groups_train` dengan `GroupKFold(5)`. Harness ini mereproduksi angka diagnostik sebelumnya sekaligus menjadi pembanding tetap. Audit data hanya melaporkan; tidak ada baris yang dihapus atau diubah.

In [14]:
# Phase 0 - harness evaluasi, reproduksi baseline, dan audit data
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from pci_ml.data import latest_checkpoint, load_checkpoint
from pci_ml.experiments import evaluate_oof, make_folds, oof_predict
from pci_ml.memory import release

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

RF_BASE_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 0.2,
    "max_samples": 0.9,
}

def rf_estimator(params=None):
    return RandomForestRegressor(random_state=SEED, **(params or RF_BASE_PARAMS))

def fit_full(params, X, y, sample_weight=None):
    model = rf_estimator(params)
    model.fit(X, y, sample_weight=sample_weight)
    return model

# Lipatan yang sama dipakai di seluruh phase.
FOLDS = make_folds(groups_train, n_splits=5)

# GroupKFold(shuffle=True) dipakai pada Phase 4 dan baru tersedia di sklearn >= 1.6.
if tuple(int(part) for part in sklearn.__version__.split(".")[:2]) < (1, 6):
    raise RuntimeError(
        "Butuh scikit-learn >= 1.6 untuk GroupKFold(shuffle=True); "
        f"versi terpasang {sklearn.__version__}."
    )

BASE_MODEL = fit_full(RF_BASE_PARAMS, X_train, y_train)
BASELINE_OOF = oof_predict(BASE_MODEL, X_train, y_train, FOLDS)
baseline_eval = evaluate_oof(y_train, BASELINE_OOF, groups_train, FOLDS)

depths = np.array([est.get_depth() for est in BASE_MODEL.estimators_])
train_mae = mean_absolute_error(y_train, BASE_MODEL.predict(X_train))
first_bin = baseline_eval["frame"]
first_bin = first_bin[(first_bin["pci"] > 0) & (first_bin["pci"] <= 25)]

print("Phase 0 - baseline")
print(f"- scikit-learn : {sklearn.__version__}")
print(f"- pooled OOF : MAE {baseline_eval['pooled']['mae']:.4f} | "
      f"RMSE {baseline_eval['pooled']['rmse']:.4f} | R2 {baseline_eval['pooled']['r2']:.4f}")
print(f"- per fold   : " + " | ".join(f"{value:.2f}" for value in baseline_eval["per_fold"]))
print(f"- train MAE  : {train_mae:.4f} (gap {baseline_eval['pooled']['mae'] - train_mae:.2f})")
print(f"- tree depth : max {depths.max()} | mean {depths.mean():.2f}")
print(f"- PCI <= 40  : n {baseline_eval['low']['n']} | MAE {baseline_eval['low']['mae']:.2f} | "
      f"bias {baseline_eval['low']['bias']:+.2f}")
print(f"- PCI > 85   : n {baseline_eval['high']['n']} | MAE {baseline_eval['high']['mae']:.2f}")
print(f"- bin (0,25] tanpa PCI=0: n {len(first_bin)} | MAE {first_bin['err'].mean():.2f} | "
      f"bias {first_bin['signed'].mean():+.2f}")

print("\nError per rentang PCI (baseline):")
display(baseline_eval["per_bin"].round(3))

print("\n10 jalan dengan MAE OOF terbesar (baseline):")
display(baseline_eval["per_road"].head(10).round(2))

# Audit jalan bermasalah dan segmen PCI = 0
audit_train, _, _, _ = load_checkpoint(filename, latest_checkpoint(filename))
feature_frame = pd.DataFrame(X_train, columns=features)
audit = baseline_eval["frame"].copy()
audit["segment"] = audit_train["SEGMENT_ID"].to_numpy()
audit["total_density"] = X_train.sum(axis=1)
audit["n_distress"] = (X_train > 0).sum(axis=1)

def top_features(row_index, k=5):
    row = feature_frame.iloc[row_index]
    row = row[row > 0].sort_values(ascending=False).head(k)
    return ", ".join(f"{name}={value:.2f}" for name, value in row.items()) or "-"

for road in ("R027", "R003", "R022", "R021"):
    part = audit[audit["road"] == road]
    print(f"\n{road}: n {len(part)} | PCI mean {part['pci'].mean():.1f} | "
          f"MAE {part['err'].mean():.1f} | bias {part['signed'].mean():+.1f}")
    worst = part.reindex(part["signed"].abs().sort_values(ascending=False).index).head(5)
    for row_index, row in worst.iterrows():
        print(f"  {row['segment']}: PCI {row['pci']:.0f} -> pred {row['pred']:.1f} "
              f"(signed {row['signed']:+.1f}) | total {row['total_density']:.2f} | "
              f"{top_features(row_index)}")

zero_pci = audit[audit["pci"] == 0]
print(f"\nSegmen PCI = 0: {len(zero_pci)}")
display(zero_pci[["road", "segment", "pred", "signed", "total_density", "n_distress"]].round(2))

low_density = audit[(audit["pci"] <= 25) & (audit["total_density"] < 0.25)]
high_density = audit[(audit["pci"] >= 80) & (audit["total_density"] >= 0.60)]
print(f"Flag: PCI <= 25 dengan total density < 0.25 -> {len(low_density)} segmen")
print(f"Flag: PCI >= 80 dengan total density >= 0.60 -> {len(high_density)} segmen")

baseline_eval.pop("frame", None)
release("audit_train", "audit", "feature_frame", "zero_pci", "low_density",
        "high_density", "worst", "part", "BASE_MODEL")

Phase 0 - baseline
- scikit-learn : 1.9.1
- pooled OOF : MAE 7.1919 | RMSE 11.3480 | R2 0.7545
- per fold   : 5.92 | 11.18 | 5.34 | 7.36 | 6.09
- train MAE  : 2.4085 (gap 4.78)
- tree depth : max 52 | mean 43.76
- PCI <= 40  : n 119 | MAE 20.55 | bias +20.01
- PCI > 85   : n 254 | MAE 2.52
- bin (0,25] tanpa PCI=0: n 24 | MAE 24.35 | bias +24.35

Error per rentang PCI (baseline):


,n,mae,bias
bin,,,
"(-0.001, 25.0]",31,31.269,31.269
"(25.0, 40.0]",88,16.774,16.041
"(40.0, 55.0]",151,9.681,8.766
"(55.0, 70.0]",186,6.331,2.600
"(70.0, 85.0]",223,4.417,-2.334
"(85.0, 100.0]",254,2.520,-1.814



10 jalan dengan MAE OOF terbesar (baseline):


,n,pci_mean,mae,bias,fold
road,,,,,
R027,10,32.25,25.54,25.54,3
R003,18,47.69,16.54,12.41,1
R022,80,52.95,14.15,12.43,2
R021,40,55.09,13.41,10.74,2
R012,40,57.65,10.77,9.60,4
R010,50,62.02,9.21,3.82,4
R007,40,55.76,8.82,5.72,1
R011,40,59.60,8.75,3.28,2
R017,40,53.42,8.39,5.73,5



R027: n 10 | PCI mean 32.3 | MAE 25.5 | bias +25.5
  S010: PCI 44 -> pred 82.8 (signed +38.3) | total 0.10 | RUTTING_H=0.08, LT_CRACK_L=0.02
  S004: PCI 17 -> pred 48.9 (signed +31.7) | total 0.42 | GATOR_CRACK_M=0.20, PATCHING_M=0.13, RUTTING_H=0.06, LT_CRACK_L=0.02, POTHOLES_L=0.01
  S008: PCI 32 -> pred 61.7 (signed +29.3) | total 0.17 | RUTTING_H=0.07, GATOR_CRACK_M=0.05, LT_CRACK_H=0.03, PATCHING_M=0.01, LT_CRACK_M=0.00
  S003: PCI 13 -> pred 39.2 (signed +25.7) | total 0.52 | PATCHING_M=0.23, GATOR_CRACK_M=0.19, RUTTING_H=0.07, LT_CRACK_M=0.03
  S001: PCI 22 -> pred 45.4 (signed +23.2) | total 0.39 | GATOR_CRACK_M=0.17, PATCHING_M=0.15, RUTTING_H=0.04, LT_CRACK_M=0.02, POTHOLES_L=0.01

R003: n 18 | PCI mean 47.7 | MAE 16.5 | bias +12.4
  S006: PCI 0 -> pred 53.0 (signed +53.0) | total 0.20 | DEPRESSION_H=0.05, LT_CRACK_M=0.04, RAVELLING_M=0.03, GATOR_CRACK_M=0.03, PATCHING_M=0.02
  S014: PCI 20 -> pred 66.6 (signed +46.6) | total 0.93 | PATCHING_H=0.93
  S013: PCI 20 -> pred 66.

,road,segment,pred,signed,total_density,n_distress
17,R003,S006,52.97,52.97,0.20,10
288,R012,S004,57.56,57.56,0.24,9
555,R021,S007,68.06,68.06,0.26,8
564,R021,S016,65.89,65.89,0.31,9
640,R022,S052,50.02,50.02,0.52,9
664,R022,S076,39.21,39.21,0.46,8
668,R022,S080,51.31,51.31,0.68,9


Flag: PCI <= 25 dengan total density < 0.25 -> 12 segmen
Flag: PCI >= 80 dengan total density >= 0.60 -> 0 segmen


**Kesimpulan Phase 0.** Harness mereproduksi baseline: pooled OOF MAE 7.1919, MAE per fold 5.92 / 11.18 / 5.34 / 7.36 / 6.09, dan bias positif besar pada PCI rendah (+31.3 pada bin pertama bila PCI = 0 disertakan; +24.3 bila tidak). Pohon tumbuh penuh (kedalaman maksimum 52, rata-rata 43.8), sehingga regularisasi bukan kunci. Audit menemukan 7 segmen PCI = 0 dengan total density 0.20-0.68 tetapi prediksi 39-68; ini kandidat label noise atau distress yang tidak terukur. Tidak ada data yang diubah.

##### 4.2 Phase 1 - Fitur agregat

PCI dibangun dari efek distress yang kurang lebih aditif, sedangkan pohon sulit menjumlahkan banyak fitur. Dua fitur baris ditambahkan: `TOTAL_DENSITY` (jumlah 55 fitur) dan `SEVERITY_WEIGHTED` (L=1, M=2, H=3). Keduanya *stateless* sehingga aman diterapkan pada train dan test, dan tetap di luar checkpoint.

In [15]:
# Phase 1 - fitur agregat
import pandas as pd
from pci_ml.experiments import aggregate_features, evaluate_oof, oof_predict
from pci_ml.memory import release
from pci_ml.random_forest import tune_random_forest

X_TRAIN_AUG, AUG_NAMES = aggregate_features(X_train, features)
X_TEST_AUG, _ = aggregate_features(X_test, features)

p1_oof = oof_predict(rf_estimator(), X_TRAIN_AUG, y_train, FOLDS)
p1_eval = evaluate_oof(y_train, p1_oof, groups_train, FOLDS)

print("Phase 1 - fitur agregat")
print(f"- pooled OOF : MAE {p1_eval['pooled']['mae']:.4f} | "
      f"RMSE {p1_eval['pooled']['rmse']:.4f} | R2 {p1_eval['pooled']['r2']:.4f}")
print(f"- per fold   : " + " | ".join(f"{value:.2f}" for value in p1_eval["per_fold"]))
print(f"- PCI <= 40  : MAE {p1_eval['low']['mae']:.2f} | bias {p1_eval['low']['bias']:+.2f}")
print(f"- PCI > 85   : MAE {p1_eval['high']['mae']:.2f}")

p1_model = fit_full(RF_BASE_PARAMS, X_TRAIN_AUG, y_train)
importance = pd.Series(p1_model.feature_importances_, index=AUG_NAMES)
print("\nImportance fitur agregat:")
print(importance[["TOTAL_DENSITY", "SEVERITY_WEIGHTED"]].round(4).to_string())
print("\nTop 10 fitur:")
print(importance.sort_values(ascending=False).head(10).round(4).to_string())

RF_AUG_SPACE = {
    "n_estimators": [300],
    "max_depth": [None, 30],
    "min_samples_split": [2, 4],
    "min_samples_leaf": [1, 2],
    "max_features": [0.1, 0.15, 0.2, 0.25, 0.3],
    "max_samples": [0.6, 0.7, 0.8, 0.9],
}
p1_tune = tune_random_forest(
    X_TRAIN_AUG, y_train, groups_train, RF_AUG_SPACE,
    n_iter=40, cv_splits=5, random_state=SEED,
)
print(f"\nRe-tune kecil: {p1_tune['params']}")
print(f"CV pooled MAE: {p1_tune['cv_pooled']['mae']:.4f}")

p1_eval.pop("frame", None)
release("p1_model", "importance", "p1_tune")

Phase 1 - fitur agregat
- pooled OOF : MAE 5.9208 | RMSE 8.9912 | R2 0.8459
- per fold   : 5.92 | 7.25 | 4.64 | 6.77 | 5.03
- PCI <= 40  : MAE 13.63 | bias +13.43
- PCI > 85   : MAE 3.22

Importance fitur agregat:
TOTAL_DENSITY        0.2162
SEVERITY_WEIGHTED    0.3347

Top 10 fitur:
SEVERITY_WEIGHTED    0.3347
TOTAL_DENSITY        0.2162
POTHOLES_H           0.0800
GATOR_CRACK_M        0.0702
GATOR_CRACK_H        0.0668
PATCHING_M           0.0314
POTHOLES_M           0.0236
POLISH_AGG           0.0176
PATCHING_H           0.0119
PATCHING_L           0.0112

Re-tune kecil: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_samples': 0.9, 'max_features': 0.25, 'max_depth': None}
CV pooled MAE: 5.8773


**Kesimpulan Phase 1.** Fitur agregat menurunkan pooled OOF MAE dari 7.1919 ke 5.9208 dan memperkecil bias PCI rendah (MAE PCI <= 40 dari 20.55 ke 13.63; bias dari +20.01 ke +13.43). Namun MAE PCI > 85 naik dari 2.52 ke 3.22 (+0.70), sedikit di atas batas 0.5. `SEVERITY_WEIGHTED` dan `TOTAL_DENSITY` menjadi dua fitur terpenting (importance 0.335 dan 0.216, total 0.55). Re-tune kecil (40 kandidat) memilih `max_features=0.25` dengan MAE 5.8773; selisih 0.04 dari 5.9208 termasuk noise, sehingga parameter baseline dipertahankan.

##### 4.3 Phase 2 - Bobot sampel untuk kondisi buruk

Sampel diberi bobot `w = 1 + alpha * (100 - PCI) / 100`, sehingga segmen dengan PCI rendah lebih berpengaruh saat latihan. Metrik evaluasi tetap tanpa bobot. `alpha = 0` berarti baseline.

In [16]:
# Phase 2 - bobot sampel
import pandas as pd
from IPython.display import display
from pci_ml.experiments import condition_weights, evaluate_oof, oof_predict

PHASE2_ALPHAS = (0, 1, 2, 4, 6)
phase2_rows = []
PHASE2_OOF = {}

for alpha in PHASE2_ALPHAS:
    weights = None if alpha == 0 else condition_weights(y_train, alpha)
    oof = oof_predict(rf_estimator(), X_train, y_train, FOLDS, sample_weight=weights)
    ev = evaluate_oof(y_train, oof, groups_train, FOLDS)
    PHASE2_OOF[alpha] = oof
    phase2_rows.append({
        "alpha": alpha,
        "OOF MAE": ev["pooled"]["mae"],
        "OOF RMSE": ev["pooled"]["rmse"],
        "OOF R2": ev["pooled"]["r2"],
        "Fold 1": ev["per_fold"][0],
        "Fold 2": ev["per_fold"][1],
        "Fold 3": ev["per_fold"][2],
        "Fold 4": ev["per_fold"][3],
        "Fold 5": ev["per_fold"][4],
        "PCI<=40 MAE": ev["low"]["mae"],
        "PCI<=40 bias": ev["low"]["bias"],
        "PCI>85 MAE": ev["high"]["mae"],
    })
    ev.pop("frame", None)

phase2_table = pd.DataFrame(phase2_rows).set_index("alpha")
print("Phase 2 - bobot sampel (fitur asli)")
display(phase2_table.round(3))

Phase 2 - bobot sampel (fitur asli)


,OOF MAE,OOF RMSE,OOF R2,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,PCI<=40 MAE,PCI<=40 bias,PCI>85 MAE
alpha,,,,,,,,,,,
0,7.192,11.348,0.755,5.924,11.185,5.335,7.357,6.094,20.550,20.008,2.520
1,6.946,10.946,0.772,5.717,10.754,5.399,6.729,6.058,19.414,18.357,2.517
2,6.839,10.809,0.777,5.620,10.505,5.286,6.795,5.924,18.752,17.565,2.520
4,6.726,10.642,0.784,5.566,10.443,5.105,6.614,5.838,18.647,17.199,2.320
6,6.680,10.594,0.786,5.438,10.330,5.135,6.642,5.790,18.427,17.027,2.299


**Kesimpulan Phase 2.** Bobot menurunkan pooled OOF MAE secara konsisten, dari 7.1919 (`alpha=0`) ke 6.7262 (`alpha=4`), dan memperbaiki MAE PCI <= 40 dari 20.55 ke 18.65 serta biasnya dari +20.01 ke +17.20. `alpha=6` hanya memberi perbaikan marginal (6.6803). MAE PCI > 85 justru membaik (2.52 ke 2.32). Efeknya lebih kecil daripada fitur agregat, tetapi arahnya benar dan tidak merugikan jalan baik.

##### 4.4 Phase 3 - Kalibrasi prediksi *post-hoc*

Kalibrator dipelajari dari prediksi *out-of-fold* data latih saja, lalu diterapkan ke lipatan yang ditahan (*leave-one-fold-out*). Dua metode diuji: regresi linear dan isotonic. Untuk model final, kalibrator dilatih pada seluruh OOF dan diterapkan ke data uji; data uji tidak pernah dipakai untuk melatih kalibrator.

In [17]:
# Phase 3 - kalibrasi post-hoc
import pandas as pd
from IPython.display import display
from pci_ml.experiments import calibrate_oof, evaluate_oof

phase3_rows = []
PHASE3_OOF = {}

for method in ("linear", "isotonic"):
    calibrated = calibrate_oof(BASELINE_OOF, y_train, FOLDS, method)
    ev = evaluate_oof(y_train, calibrated, groups_train, FOLDS)
    PHASE3_OOF[method] = calibrated
    phase3_rows.append({
        "method": method,
        "OOF MAE": ev["pooled"]["mae"],
        "OOF RMSE": ev["pooled"]["rmse"],
        "OOF R2": ev["pooled"]["r2"],
        "PCI<=40 MAE": ev["low"]["mae"],
        "PCI<=40 bias": ev["low"]["bias"],
        "PCI>85 MAE": ev["high"]["mae"],
    })
    ev.pop("frame", None)

phase3_table = pd.DataFrame(phase3_rows).set_index("method")
print("Phase 3 - kalibrasi pada baseline")
display(phase3_table.round(3))

Phase 3 - kalibrasi pada baseline


,OOF MAE,OOF RMSE,OOF R2,PCI<=40 MAE,PCI<=40 bias,PCI>85 MAE
method,,,,,,
linear,7.438,10.715,0.781,15.321,13.289,3.667
isotonic,7.513,11.034,0.768,15.044,12.763,2.598


**Kesimpulan Phase 3.** Kalibrasi isotonic memperbaiki R2 (0.7545 ke 0.7679) dan menurunkan bias PCI rendah (MAE PCI <= 40 dari 20.55 ke 15.04; bias dari +20.01 ke +12.76), tetapi pooled OOF MAE justru naik ke 7.5129 dan MAE PCI > 85 naik ke 2.60. Kalibrasi linear mirip (MAE 7.4376, PCI > 85 3.67). Kalibrasi saja tidak cukup: ia memperbaiki arah bias tetapi tidak menambah informasi fitur. Karena itu kalibrasi diuji sebagai pelengkap fitur agregat.

##### 4.5 Phase 4 - Perbandingan, uji ketahanan, dan rekomendasi

Kandidat dipilih hanya dari CV. Data uji dilaporkan untuk finalis sebagai gambaran akhir, bukan untuk memilih konfigurasi. Uji ketahanan mengulang CV dengan lima pengacakan jalan ke lipatan (`GroupKFold(shuffle=True)`), lalu melaporkan rata-rata dan simpangan baku pooled MAE.

In [18]:
# Phase 4 - perbandingan, uji ketahanan, dan evaluasi test
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pci_ml.experiments import (apply_calibrator, calibrate_oof,
                                condition_weights, evaluate_oof,
                                fit_final_calibrator, make_folds, oof_predict)
from pci_ml.memory import release

def evaluate_variant(X_tr, X_te, params, alpha=0, calibrate=None, with_test=True):
    weights = None if alpha == 0 else condition_weights(y_train, alpha)
    raw_oof = oof_predict(rf_estimator(params), X_tr, y_train, FOLDS,
                          sample_weight=weights)
    oof = raw_oof if calibrate is None else calibrate_oof(
        raw_oof, y_train, FOLDS, calibrate)
    ev = evaluate_oof(y_train, oof, groups_train, FOLDS)
    ev["oof"] = oof

    if with_test:
        model = fit_full(params, X_tr, y_train, sample_weight=weights)
        predictions = model.predict(X_te)
        if calibrate is not None:
            calibrator = fit_final_calibrator(raw_oof, y_train, calibrate)
            predictions = apply_calibrator(calibrator, predictions, calibrate)
        ev["test"] = {
            "mae": float(mean_absolute_error(y_test, predictions)),
            "rmse": float(np.sqrt(mean_squared_error(y_test, predictions))),
            "r2": float(r2_score(y_test, predictions)),
        }
    else:
        ev["test"] = {"mae": np.nan, "rmse": np.nan, "r2": np.nan}
    return ev

variants = {
    "Baseline": evaluate_variant(X_train, X_test, RF_BASE_PARAMS),
    "Phase 1 (agregat)": evaluate_variant(X_TRAIN_AUG, X_TEST_AUG, RF_BASE_PARAMS),
    "Phase 2 (bobot a=4)": evaluate_variant(X_train, X_test, RF_BASE_PARAMS, alpha=4),
    "Phase 3 (isotonic)": evaluate_variant(X_train, X_test, RF_BASE_PARAMS,
                                           calibrate="isotonic", with_test=False),
    "Kombinasi (agregat + bobot a=4 + isotonic)": evaluate_variant(
        X_TRAIN_AUG, X_TEST_AUG, RF_BASE_PARAMS, alpha=4, calibrate="isotonic"
    ),
}

rows = []
for name, ev in variants.items():
    rows.append({
        "Kandidat": name,
        "OOF MAE": ev["pooled"]["mae"],
        "OOF RMSE": ev["pooled"]["rmse"],
        "OOF R2": ev["pooled"]["r2"],
        "Fold 1": ev["per_fold"][0],
        "Fold 2": ev["per_fold"][1],
        "Fold 3": ev["per_fold"][2],
        "Fold 4": ev["per_fold"][3],
        "Fold 5": ev["per_fold"][4],
        "PCI<=40 MAE": ev["low"]["mae"],
        "PCI<=40 bias": ev["low"]["bias"],
        "PCI>85 MAE": ev["high"]["mae"],
        "Test MAE": ev["test"]["mae"],
        "Test RMSE": ev["test"]["rmse"],
        "Test R2": ev["test"]["r2"],
    })
comparison = pd.DataFrame(rows).set_index("Kandidat")
print("Phase 4 - perbandingan kandidat")
display(comparison.round(3))

# Uji ketahanan: 5 pengacakan jalan ke lipatan
def shuffled_cv_mae(X_tr, params, alpha=0, calibrate=None, seeds=range(5)):
    values = []
    for seed in seeds:
        folds = make_folds(groups_train, n_splits=5, shuffle=True, random_state=seed)
        weights = None if alpha == 0 else condition_weights(y_train, alpha)
        oof = oof_predict(rf_estimator(params), X_tr, y_train, folds,
                          sample_weight=weights)
        if calibrate is not None:
            oof = calibrate_oof(oof, y_train, folds, calibrate)
        values.append(mean_absolute_error(y_train, oof))
    return np.array(values)

robust_rows = []
for name, X_tr, alpha, calibrate in (
    ("Baseline", X_train, 0, None),
    ("Phase 1 (agregat)", X_TRAIN_AUG, 0, None),
    ("Phase 2 (bobot a=4)", X_train, 4, None),
    ("Kombinasi", X_TRAIN_AUG, 4, "isotonic"),
):
    values = shuffled_cv_mae(X_tr, RF_BASE_PARAMS, alpha=alpha, calibrate=calibrate)
    robust_rows.append({
        "Kandidat": name,
        "MAE mean": values.mean(),
        "MAE std": values.std(),
        "MAE min": values.min(),
        "MAE max": values.max(),
    })
robustness = pd.DataFrame(robust_rows).set_index("Kandidat")
print("\nUji ketahanan (5 pengacakan jalan ke lipatan):")
display(robustness.round(4))

# Sanity check opsional: apakah bias PCI rendah juga muncul di XGBoost?
try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1,
    )
    xgb_oof = oof_predict(xgb, X_train, y_train, FOLDS)
    xgb_eval = evaluate_oof(y_train, xgb_oof, groups_train, FOLDS)
    print("\nSanity check XGBoost:")
    print(f"- OOF MAE {xgb_eval['pooled']['mae']:.4f} | "
          f"PCI<=40 bias {xgb_eval['low']['bias']:+.2f} | "
          f"PCI>85 MAE {xgb_eval['high']['mae']:.2f}")
    xgb_eval.pop("frame", None)
except ImportError:
    print("\nXGBoost tidak terpasang; sanity check dilewati.")

release("variants", "PHASE2_OOF", "PHASE3_OOF", "robust_rows", "rows")

Phase 4 - perbandingan kandidat


,OOF MAE,OOF RMSE,OOF R2,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,PCI<=40 MAE,PCI<=40 bias,PCI>85 MAE,Test MAE,Test RMSE,Test R2
Kandidat,,,,,,,,,,,,,,
Baseline,7.192,11.348,0.755,5.924,11.185,5.335,7.357,6.094,20.550,20.008,2.520,5.753,8.114,0.866
Phase 1 (agregat),5.921,8.991,0.846,5.917,7.250,4.643,6.774,5.027,13.633,13.431,3.224,4.768,7.974,0.871
Phase 2 (bobot a=4),6.726,10.642,0.784,5.566,10.443,5.105,6.614,5.838,18.647,17.199,2.320,5.510,7.977,0.870
Phase 3 (isotonic),7.513,11.034,0.768,6.176,9.940,6.074,7.907,7.429,15.044,12.763,2.598,NaN,NaN,NaN
Kombinasi (agregat + bobot a=4 + isotonic),5.924,8.967,0.847,5.375,7.166,4.702,7.488,4.897,10.648,7.825,2.966,4.868,7.449,0.887



Uji ketahanan (5 pengacakan jalan ke lipatan):


,MAE mean,MAE std,MAE min,MAE max
Kandidat,,,,
Baseline,7.0966,0.1037,6.9286,7.2340
Phase 1 (agregat),6.0155,0.0553,5.9139,6.0642
Phase 2 (bobot a=4),6.6237,0.0610,6.5215,6.7097
Kombinasi,6.0331,0.1267,5.8087,6.1840



XGBoost tidak terpasang; sanity check dilewati.


**Kesimpulan Phase 4 dan rekomendasi.**

| Perbaikan | OOF MAE | Bias PCI <= 40 | MAE PCI > 85 | Test MAE |
|---|---:|---:|---:|---:|
| Baseline | 7.1919 | +20.01 | 2.52 | 5.753 |
| Phase 1 - agregat | 5.9208 | +13.43 | 3.22 | 4.768 |
| Phase 2 - bobot a=4 | 6.7262 | +17.20 | 2.32 | 5.510 |
| Phase 3 - isotonic | 7.5129 | +12.76 | 2.60 | - |
| Kombinasi | 5.9242 | +7.83 | 2.97 | 4.868 |

- **Fitur agregat** adalah perbaikan terbesar: pooled OOF MAE 7.19 ke 5.92. Biayanya MAE PCI > 85 naik 0.70.
- **Bobot sampel `alpha=4`** menurunkan MAE ke 6.73 dan memperbaiki bias PCI rendah, tetapi MAE PCI > 85 tetap baik (2.32). Efeknya lebih kecil daripada fitur agregat.
- **Kalibrasi isotonic** sendirian tidak memperbaiki MAE, tetapi sebagai pelengkap kombinasi ia menurunkan MAE PCI > 85 ke 2.97 (+0.45, masih dalam batas) dan bias PCI <= 40 ke +7.83.
- **Kombinasi terbaik** yang memenuhi seluruh kriteria: fitur agregat + bobot `alpha=4` + kalibrasi isotonic. Pooled OOF MAE 5.9242, 4 dari 5 fold membaik (hanya fold 4 naik 7.36 ke 7.49), bias PCI rendah menyusut jelas, dan MAE PCI > 85 hanya naik 0.45.
- **Uji ketahanan** (5 pengacakan jalan): baseline 7.097 +/- 0.104, Phase 1 6.016 +/- 0.055, Phase 2 6.624 +/- 0.061, kombinasi 6.033 +/- 0.127. Kombinasi dan Phase 1 praktis setara; kombinasi menukar sedikit MAE rata-rata dengan bias PCI rendah yang jauh lebih baik.
- Re-tune pada fitur agregat tidak memberi keuntungan di luar noise, jadi hyperparameter baseline dipertahankan.

**Keterbatasan.** Hanya 27 jalan latih dan 7 jalan uji. Variasi antar-pengacakan lipatan sekitar +/-0.06-0.13 MAE, sehingga selisih di bawah ~0.2 dianggap noise. Data uji kemungkinan lebih mudah daripada CV, jadi angka uji cenderung optimistis. Tujuh segmen PCI = 0 dengan kepadatan rendah perlu diverifikasi ke lapangan; jangan dihapus tanpa bukti.

### ***Source code* modul dan fungsi**

In [9]:
# import os
# from pathlib import Path
# from IPython.display import display, Markdown

# # Path to your src directory
# src_dir = Path("src")

# # Loop through all Python files and render them as Markdown code blocks
# for filepath in src_dir.rglob("*.py"): # rglob searches subdirectories too
    
#     # Ignore any files inside Jupyter checkpoint folders
#     if ".ipynb_checkpoints" in filepath.parts:
#         continue

#     with open(filepath, "r", encoding="utf-8") as f:
#         code = f.read()
    
#     # Creates a nice header for the filename and formats the code
#     display(Markdown(f"#### `{filepath.name}`\n```python\n{code}\n```\n---"))